# comfy_colab — SDXL images + Wan2.2 video on a free Colab T4

One self-contained notebook. **Runtime → Change runtime type → T4 GPU**, then run cells top-to-bottom once (setup + server), then re-run any **Generate** cell on demand to iterate.

| Cell | Run |
|------|-----|
| 1 Setup | once |
| 2 Video models | once (only if you want video) |
| 3 Start server | once |
| 3b Tunnel GUI | after server, per session |
| 4 Generate · SDXL image | on demand |
| 5 Generate · Wan2.2 video | on demand |
| 6 Display | after a generate cell |
| 7 Generate · Photo->Video | on demand |
| 8 Generate · Persona anchor | once per persona |
| 9 Generate · Identity scene | per scene |

Images/clips are saved to `ComfyUI/output/` on the VM (ephemeral) — download via the file browser, or use the Drive mount cell to persist.


In [ ]:
#@title 1 · Setup: ComfyUI + SDXL + GGUF node { display-mode: "form" }
import os, subprocess
COMFY = "/content/ComfyUI"
CKPT_DIR = f"{COMFY}/models/checkpoints"
CKPT = f"{CKPT_DIR}/sd_xl_base_1.0.safetensors"
SDXL_URL = ("https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/"
            "resolve/main/sd_xl_base_1.0.safetensors")

if not os.path.isdir(f"{COMFY}/.git"):
    print("Cloning ComfyUI...")
    subprocess.run(["git","clone","-q","https://github.com/comfyanonymous/ComfyUI.git", COMFY], check=True)
else:
    print("ComfyUI present")
print("Installing requirements...")
subprocess.run(["pip","install","-q","-r",f"{COMFY}/requirements.txt"], check=True)

# GGUF custom node (for Wan2.2 quantized model)
GGUF_NODE = f"{COMFY}/custom_nodes/ComfyUI-GGUF"
if not os.path.isdir(f"{GGUF_NODE}/.git"):
    print("Installing ComfyUI-GGUF node...")
    subprocess.run(["git","clone","-q","https://github.com/city96/ComfyUI-GGUF.git", GGUF_NODE], check=True)
print("Installing gguf (python dep for ComfyUI-GGUF)...")
subprocess.run(["pip","install","-q","gguf"], check=True)

if not os.path.exists(CKPT):
    print("Downloading SDXL base (~6.5 GB)...")
    os.makedirs(CKPT_DIR, exist_ok=True)
    subprocess.run(["wget","-q","--show-progress","-O",CKPT,SDXL_URL], check=True)
print(f"\nSetup done. SDXL: {os.path.getsize(CKPT)//1048576} MB")


In [ ]:
#@title 2 · (Optional) Download Wan2.2 5B video models { display-mode: "form" }
import os, subprocess
DIFF = f"{COMFY}/models/diffusion_models"
TE   = f"{COMFY}/models/text_encoders"
VAE  = f"{COMFY}/models/vae"
for d in (DIFF, TE, VAE): os.makedirs(d, exist_ok=True)

FILES = {
    # Wan2.2 TI2V 5B, GGUF Q4_K_M (~3.3 GB) — fits T4 VRAM comfortably
    f"{DIFF}/wan2.2_ti2v_5B-Q4_K_M.gguf":
        "https://huggingface.co/QuantStack/Wan2.2-TI2V-5B-GGUF/resolve/main/Wan2.2-TI2V-5B-Q4_K_M.gguf",
    # text encoder (umt5 fp8, ~6.4 GB) — freed after encode
    f"{TE}/umt5_xxl_fp8_e4m3fn_scaled.safetensors":
        "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    # Wan2.2 VAE (~1.3 GB)
    f"{VAE}/wan2.2_vae.safetensors":
        "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan2.2_vae.safetensors",
}
for path, url in FILES.items():
    if os.path.exists(path):
        print(f"  exists: {os.path.basename(path)} ({os.path.getsize(path)//1048576} MB)")
    else:
        print(f"  downloading {os.path.basename(path)} ...")
        subprocess.run(["wget","-q","--show-progress","-O",path,url], check=True)
print("\nVideo models ready.")


In [ ]:
#@title 3 · Start ComfyUI server { display-mode: "form" }
import subprocess, time, os, urllib.request
PORT = 8188
PIDFILE = "/content/comfy.pid"; LOGFILE = "/content/comfy.log"
def _alive(pid):
    try: os.kill(pid, 0); return True
    except Exception: return False
pid = None
if os.path.exists(PIDFILE):
    try: pid = int(open(PIDFILE).read().strip())
    except: pid = None
if pid and _alive(pid):
    print(f"server already running, pid={pid}")
else:
    print("launching server...")
    log = open(LOGFILE, "w")
    subprocess.Popen(["python","main.py","--listen","0.0.0.0","--port",str(PORT),
        "--output-directory", os.path.join(COMFY,"output")],
        cwd=COMFY, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    for i in range(120):
        time.sleep(1)
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=1)
            print(f"READY on http://127.0.0.1:{PORT}  (boot {i+1}s)"); break
        except Exception: pass
    else:
        print("TIMEOUT:\n"+open(LOGFILE).read()[-1500:])


In [ ]:
#@title 3b · Tunnel ComfyUI GUI to your local browser { display-mode: "form" }
#@markdown After cell 3 started the server on :8188, run this for a clickable URL
#@markdown to the full ComfyUI web GUI running on the T4. Edit sampler / CFG /
#@markdown steps / latent size / VAE / masks / ControlNet visually, then Queue —
#@markdown it runs on Colab and the image streams back. No local GPU needed.
#@markdown
#@markdown The URL is ephemeral (new every Colab session). To make a workflow
#@markdown reproducible: GUI Menu → Save (API Format) → that JSON is exactly what
#@markdown cells 4/5 POST to /prompt → paste it into a cell or commit it.
#@markdown
#@markdown proxyPort is the default (zero install). If it ever returns 502 or the
#@markdown GUI stops updating, set use_cloudflared=True and re-run.
use_cloudflared = False #@param {type:"boolean"}

import os, time, re, subprocess
PORT = 8188

if not use_cloudflared:
    from google.colab.output import eval_js
    url = eval_js(f"google.colab.kernel.proxyPort({PORT}, {{'cache': false}})")
    print("ComfyUI GUI (proxyPort — no install):")
    print(url)
    print("\nOpen this URL in your local browser. If 502 / stale, re-run with use_cloudflared=True.")
else:
    CF = "/content/cloudflared"
    if not (os.path.exists(CF) and os.access(CF, os.X_OK)):
        print("Downloading cloudflared...")
        subprocess.run(["wget", "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O", CF], check=True)
        os.chmod(CF, 0o755)
    log = open("/content/cf.log", "w")
    subprocess.Popen([CF, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    url = None
    for _ in range(40):
        time.sleep(1)
        try:
            txt = open("/content/cf.log").read()
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
            if m:
                url = m.group(0); break
        except Exception:
            pass
    if url:
        print("ComfyUI GUI (cloudflared):")
        print(url)
    else:
        print("cloudflared did not return a URL in time — see /content/cf.log")


In [ ]:
#@title 4 · Generate · SDXL image { display-mode: "form" }
prompt = "A tilted brass arrow mounted on a wooden base casts a crisp shadow onto a horizontal brass rail beneath it. Cobblestone workshop floor, copper pipes in background, soft window light from the left. Photorealistic steampunk still life, warm tones, detailed metal textures." #@param {type:"string"}
negative = "blurry, low quality, deformed, watermark, text" #@param {type:"string"}
seed = 7 #@param {type:"integer"}
import json, time, urllib.request, urllib.parse, os
SERVER = f"http://127.0.0.1:{PORT}"; OUT = f"{COMFY}/output"; os.makedirs(OUT, exist_ok=True)
wf = {"prompt":{
  "4":{"class_type":"CheckpointLoaderSimple","inputs":{"ckpt_name":"sd_xl_base_1.0.safetensors"}},
  "5":{"class_type":"EmptyLatentImage","inputs":{"width":1024,"height":1024,"batch_size":1}},
  "6":{"class_type":"CLIPTextEncode","inputs":{"text":prompt,"clip":["4",1]}},
  "7":{"class_type":"CLIPTextEncode","inputs":{"text":negative,"clip":["4",1]}},
  "3":{"class_type":"KSampler","inputs":{"seed":seed,"steps":25,"cfg":7.5,"sampler_name":"euler","scheduler":"normal","denoise":1.0,"model":["4",0],"positive":["6",0],"negative":["7",0],"latent_image":["5",0]}},
  "8":{"class_type":"VAEDecode","inputs":{"samples":["3",0],"vae":["4",2]}},
  "9":{"class_type":"SaveImage","inputs":{"filename_prefix":"sdxl","images":["8",0]}}}}
pid = json.loads(urllib.request.urlopen(urllib.request.Request(f"{SERVER}/prompt",
    data=json.dumps(wf).encode(), headers={"Content-Type":"application/json"})).read())["prompt_id"]
print("submitted:", pid)
t0=time.time(); LAST=None
while time.time()-t0<300:
    try: h=json.loads(urllib.request.urlopen(f"{SERVER}/history/{pid}").read())
    except: h={}
    if pid in h:
        for _,out in h[pid]["outputs"].items():
            for it in out.get("images",[]):
                q=urllib.parse.urlencode({"filename":it["filename"],"subfolder":it.get("subfolder",""),"type":"output"})
                data=urllib.request.urlopen(f"{SERVER}/view?{q}").read()
                dest=os.path.join(OUT,it["filename"]); open(dest,"wb").write(data); LAST=dest
                print(f"DONE in {time.time()-t0:.1f}s -> {dest} ({len(data)//1024} KB)")
        break
    time.sleep(1)
else: print("TIMEOUT")


In [ ]:
#@title 5 · Generate · Wan2.2 video { display-mode: "form" }
prompt = "A man grips the wooden handles of a loaded handcart and lifts them up, a small elephant standing on the flatbed of the cart, the cart tilts back onto its wheels, dusty village lane, warm golden afternoon light, realistic motion, cinematic, photorealistic" #@param {type:"string"}
negative = "色调艳丽，过曝，静态，细节模糊不清，字幕，静止，低质量，变形，水印" #@param {type:"string"}
seed = 7 #@param {type:"integer"}
width = 832 #@param {type:"integer"}
height = 480 #@param {type:"integer"}
frames = 49 #@param {type:"integer"}
steps = 40 #@param {type:"integer"}
import json, time, urllib.request, urllib.parse, os
SERVER = f"http://127.0.0.1:{PORT}"; OUT = f"{COMFY}/output"; os.makedirs(OUT, exist_ok=True)
wf = {"prompt":{
  "37":{"class_type":"UnetLoaderGGUF","inputs":{"unet_name":"wan2.2_ti2v_5B-Q4_K_M.gguf"}},
  "38":{"class_type":"CLIPLoader","inputs":{"clip_name":"umt5_xxl_fp8_e4m3fn_scaled.safetensors","type":"wan"}},
  "39":{"class_type":"VAELoader","inputs":{"vae_name":"wan2.2_vae.safetensors"}},
  "48":{"class_type":"ModelSamplingSD3","inputs":{"model":["37",0],"shift":8.0}},
  "6":{"class_type":"CLIPTextEncode","inputs":{"text":prompt,"clip":["38",0]}},
  "7":{"class_type":"CLIPTextEncode","inputs":{"text":negative,"clip":["38",0]}},
  "55":{"class_type":"Wan22ImageToVideoLatent","inputs":{"vae":["39",0],"width":width,"height":height,"length":frames,"batch_size":1}},
  "3":{"class_type":"KSampler","inputs":{"seed":seed,"steps":steps,"cfg":5.0,"sampler_name":"uni_pc","scheduler":"simple","denoise":1.0,"model":["48",0],"positive":["6",0],"negative":["7",0],"latent_image":["55",0]}},
  "8":{"class_type":"VAEDecode","inputs":{"samples":["3",0],"vae":["39",0]}},
  "47":{"class_type":"SaveWEBM","inputs":{"images":["8",0],"filename_prefix":"wan22","codec":"vp9","fps":16,"crf":20}}}}
pid = json.loads(urllib.request.urlopen(urllib.request.Request(f"{SERVER}/prompt",
    data=json.dumps(wf).encode(), headers={"Content-Type":"application/json"})).read())["prompt_id"]
print("submitted:", pid, f"(this can take several minutes — {frames} frames)")
t0=time.time(); LAST=None
while time.time()-t0<900:
    try: h=json.loads(urllib.request.urlopen(f"{SERVER}/history/{pid}").read())
    except: h={}
    if pid in h:
        for _,out in h[pid]["outputs"].items():
            for kind in ("gifs","images","videos"):
                for it in out.get(kind,[]):
                    q=urllib.parse.urlencode({"filename":it["filename"],"subfolder":it.get("subfolder",""),"type":it.get("type","output")})
                    data=urllib.request.urlopen(f"{SERVER}/view?{q}").read()
                    dest=os.path.join(OUT,it["filename"]); open(dest,"wb").write(data); LAST=dest
                    print(f"DONE in {time.time()-t0:.1f}s -> {dest} ({len(data)//1024} KB)")
        # also check for execution error
        if h[pid].get("status",{}).get("status_str")=="error":
            print("SERVER ERROR:", json.dumps(h[pid]["status"].get("messages",[]))[:500])
        break
    time.sleep(2)
else: print("TIMEOUT")


In [ ]:
#@title 6 · Display last result { display-mode: "form" }
from IPython.display import Image, Video, HTML, display
import os
if LAST:
    ext = os.path.splitext(LAST)[1].lower()
    if ext in (".webm",".mp4"):
        display(HTML(f'''<video controls src="data:video/webm;base64,{__import__('base64').b64encode(open(LAST,'rb').read()).decode()}"></video>'''))
    else:
        display(Image(filename=LAST))
else:
    print("nothing yet — run a Generate cell")


In [ ]:
#@title 7 · Generate · Photo->Video (Wan2.2 I2V, N personas) { display-mode: "form" }
#@markdown Conditions motion on a START PHOTO (the persona as frame 1). One prompt
#@markdown produces one video per photo. `photos` = comma-separated filenames
#@markdown already in ComfyUI/input/ (pi uploads them via the tunnel first).
#@markdown Leave `photos` blank to be prompted to upload manually.
photos = "persona_scene.png" #@param {type:"string"}
prompt = "ayse odasinda ders calisiyor, defterini karalar, kalemini bicimler, derse dalmis, sonra kamera'ya bakar ve gulumser, gercekci, sinematik, yumusak pencere isigi, detayli" #@param {type:"string"}
negative = "色调艳丽, 过曝, 静态, 细节模糊不清, 字幕, 静止, 低质量, 变形, 水印, deformed, extra fingers" #@param {type:"string"}
seed = 7 #@param {type:"integer"}
width = 832 #@param {type:"integer"}
height = 480 #@param {type:"integer"}
frames = 49 #@param {type:"integer"}
steps = 40 #@param {type:"integer"}
import os, json, time, urllib.request, urllib.parse
SERVER = f"http://127.0.0.1:{PORT}"; OUT = f"{COMFY}/output"; INP = f"{COMFY}/input"; os.makedirs(OUT, exist_ok=True)

names = [n.strip() for n in photos.split(",") if n.strip()]
if not names:
    from google.colab import files
    up = files.upload()
    for fn, data in up.items():
        open(os.path.join(INP, fn), "wb").write(data)
    names = list(up.keys())
assert names, "no photos provided"
print("personas:", names)

def run_i2v(image_name, tag):
    wf = {"prompt":{
      "37":{"class_type":"UnetLoaderGGUF","inputs":{"unet_name":"wan2.2_ti2v_5B-Q4_K_M.gguf"}},
      "38":{"class_type":"CLIPLoader","inputs":{"clip_name":"umt5_xxl_fp8_e4m3fn_scaled.safetensors","type":"wan"}},
      "39":{"class_type":"VAELoader","inputs":{"vae_name":"wan2.2_vae.safetensors"}},
      "48":{"class_type":"ModelSamplingSD3","inputs":{"model":["37",0],"shift":8.0}},
      "6":{"class_type":"CLIPTextEncode","inputs":{"text":prompt,"clip":["38",0]}},
      "7":{"class_type":"CLIPTextEncode","inputs":{"text":negative,"clip":["38",0]}},
      "57":{"class_type":"LoadImage","inputs":{"image":image_name}},
      "55":{"class_type":"Wan22ImageToVideoLatent","inputs":{"vae":["39",0],"start_image":["57",0],"width":width,"height":height,"length":frames,"batch_size":1}},
      "3":{"class_type":"KSampler","inputs":{"seed":seed,"steps":steps,"cfg":5.0,"sampler_name":"uni_pc","scheduler":"simple","denoise":1.0,"model":["48",0],"positive":["6",0],"negative":["7",0],"latent_image":["55",0]}},
      "8":{"class_type":"VAEDecode","inputs":{"samples":["3",0],"vae":["39",0]}},
      "47":{"class_type":"SaveWEBM","inputs":{"images":["8",0],"filename_prefix":tag,"codec":"vp9","fps":16,"crf":20}}}}
    pid = json.loads(urllib.request.urlopen(urllib.request.Request(f"{SERVER}/prompt",
        data=json.dumps(wf).encode(), headers={"Content-Type":"application/json"})).read())["prompt_id"]
    print(f"[{tag}] submitted {pid} ({frames}f, can take minutes)")
    t0 = time.time()
    while time.time()-t0 < 900:
        try: h = json.loads(urllib.request.urlopen(f"{SERVER}/history/{pid}").read())
        except: h = {}
        if pid in h:
            if h[pid].get("status",{}).get("status_str") == "error":
                print(f"[{tag}] SERVER ERROR:", json.dumps(h[pid]["status"].get("messages",[]))[:400]); return None
            for _, out in h[pid]["outputs"].items():
                for kind in ("gifs","images","videos"):
                    for it in out.get(kind,[]):
                        q = urllib.parse.urlencode({"filename":it["filename"],"subfolder":it.get("subfolder",""),"type":it.get("type","output")})
                        data = urllib.request.urlopen(f"{SERVER}/view?{q}").read()
                        dest = os.path.join(OUT, it["filename"]); open(dest,"wb").write(data)
                        print(f"[{tag}] DONE {time.time()-t0:.1f}s -> {dest} ({len(data)//1024} KB)")
                        return dest
            return None
        time.sleep(2)
    print(f"[{tag}] TIMEOUT"); return None

LAST = None; results = []
for i, nm in enumerate(names, 1):
    tag = f"wan22_i2v_{i}"
    r = run_i2v(nm, tag)
    if r:
        results.append(r); LAST = r
        try:
            from google.colab import files as _f
            _f.download(r)
        except Exception as e:
            print("  (auto-download skipped:", e, ")")
print("\nRESULTS:", results)


In [ ]:
#@title 8 · Generate · Persona anchor (FLUX.1-dev GGUF, T2I) { display-mode: "form" }
#@markdown Create the identity anchor image for a persona. Run ONCE per persona.
#@markdown Output -> output/ AND copied to input/persona_ref.png (used by cells 9 & 7).
#@markdown FLUX is English; keep the prompt in English.
persona = "Photorealistic head-and-shoulders portrait of a young woman, facing the camera, neutral soft-grey background, even soft studio lighting, sharp focus, natural skin texture. Young woman with a balanced oval face leaning slightly toward heart-shaped, face length about 1.4x width. Medium-height forehead, smooth hairline, and softly tapered jaw ending in a rounded chin. Cheekbones are moderately high and gently projected, creating subtle facial definition without sharp angles. Eyes are large almond-shaped hazel-green with slightly elongated outer corners and a deep limbal ring; spacing is approximately one eye-width. Eyebrows are dark brown, thick, naturally dense, mostly straight with a gentle arch beginning at the outer third. Nose is straight and narrow with a smooth bridge, refined tip, and small nostrils. Lips are medium-full with a softly defined cupid's bow, fuller lower lip, and relaxed neutral expression. Skin is light olive with warm undertones and faint freckles across the nose and upper cheeks. Dark brown hair is worn in a low ponytail with loose face-framing strands. Overall appearance is soft, symmetrical, and natural." #@param {type:"string"}
width = 1024 #@param {type:"integer"}
height = 1024 #@param {type:"integer"}
seed = 7 #@param {type:"integer"}
steps = 20 #@param {type:"integer"}
guidance = 3.5 #@param {type:"number"}
import os, json, time, subprocess, urllib.request, urllib.parse, shutil
SERVER = f"http://127.0.0.1:{PORT}"; COMFY = "/content/ComfyUI"
DIFF=f"{COMFY}/models/diffusion_models"; TE=f"{COMFY}/models/text_encoders"; VAE=f"{COMFY}/models/vae"
for d in (DIFF,TE,VAE): os.makedirs(d, exist_ok=True)
FILES = {
    f"{DIFF}/flux1-dev-Q4_K_S.gguf":"https://huggingface.co/Sanami/flux1-dev-gguf/resolve/main/flux1-dev-Q4_K_S.gguf",
    f"{TE}/t5xxl_fp8_e4m3fn.safetensors":"https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors",
    f"{TE}/clip_l.safetensors":"https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors",
    f"{VAE}/ae.safetensors":"https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors",
}
for p,url in FILES.items():
    if os.path.exists(p): print(f"  exists: {os.path.basename(p)} ({os.path.getsize(p)//1048576} MB)")
    else:
        print(f"  downloading {os.path.basename(p)} ...")
        subprocess.run(["wget","-q","--show-progress","-O",p,url], check=True)
wf = {"prompt":{
  "12":{"class_type":"UnetLoaderGGUF","inputs":{"unet_name":"flux1-dev-Q4_K_S.gguf"}},
  "11":{"class_type":"DualCLIPLoader","inputs":{"clip_name1":"t5xxl_fp8_e4m3fn.safetensors","clip_name2":"clip_l.safetensors","type":"flux"}},
  "10":{"class_type":"VAELoader","inputs":{"vae_name":"ae.safetensors"}},
  "6":{"class_type":"CLIPTextEncode","inputs":{"text":persona,"clip":["11",0]}},
  "26":{"class_type":"FluxGuidance","inputs":{"conditioning":["6",0],"guidance":guidance}},
  "27":{"class_type":"EmptySD3LatentImage","inputs":{"width":width,"height":height,"batch_size":1}},
  "30":{"class_type":"ModelSamplingFlux","inputs":{"model":["12",0],"max_shift":1.15,"base_shift":0.5,"width":width,"height":height}},
  "25":{"class_type":"RandomNoise","inputs":{"noise_seed":seed}},
  "16":{"class_type":"KSamplerSelect","inputs":{"sampler_name":"euler"}},
  "17":{"class_type":"BasicScheduler","inputs":{"model":["30",0],"scheduler":"simple","steps":steps,"denoise":1.0}},
  "22":{"class_type":"BasicGuider","inputs":{"model":["30",0],"conditioning":["26",0]}},
  "13":{"class_type":"SamplerCustomAdvanced","inputs":{"noise":["25",0],"guider":["22",0],"sampler":["16",0],"sigmas":["17",0],"latent_image":["27",0]}},
  "8":{"class_type":"VAEDecode","inputs":{"samples":["13",0],"vae":["10",0]}},
  "9":{"class_type":"SaveImage","inputs":{"filename_prefix":"persona","images":["8",0]}}
}}
pid=json.loads(urllib.request.urlopen(urllib.request.Request(f"{SERVER}/prompt",data=json.dumps(wf).encode(),headers={"Content-Type":"application/json"})).read())["prompt_id"]
print("submitted:",pid)
t0=time.time(); OUT=f"{COMFY}/output"; INP=f"{COMFY}/input"; os.makedirs(INP,exist_ok=True); LAST=None
while time.time()-t0<600:
    try: h=json.loads(urllib.request.urlopen(f"{SERVER}/history/{pid}").read())
    except: h={}
    if pid in h:
        if h[pid].get("status",{}).get("status_str")=="error": print("ERROR:",json.dumps(h[pid]["status"].get("messages",[]))[:500])
        for _,out in h[pid]["outputs"].items():
            for it in out.get("images",[]):
                q=urllib.parse.urlencode({"filename":it["filename"],"subfolder":it.get("subfolder",""),"type":"output"})
                data=urllib.request.urlopen(f"{SERVER}/view?{q}").read()
                dest=os.path.join(OUT,it["filename"]); open(dest,"wb").write(data); LAST=dest
                shutil.copy(dest, os.path.join(INP,"persona_ref.png"))
                print(f"DONE {time.time()-t0:.1f}s -> {dest} (also saved as input/persona_ref.png)")
        break
    time.sleep(2)
else: print("TIMEOUT")


In [ ]:
#@title 9 · Generate · Identity scene (FLUX.1-Kontext GGUF, edit) { display-mode: "form" }
#@markdown Put the persona into a NEW scene while preserving her identity (face).
#@markdown Always edits input/persona_ref.png (the cell-8 anchor). Re-run for each scene.
#@markdown Output -> output/ AND copied to input/persona_scene.png (used by cell 7).
#@markdown Kontext is English-only: write scene_prompt in English.
scene_prompt = "the same young woman is studying at her desk in a cozy room, writing in a notebook, then looks up at the camera and smiles, soft warm afternoon window light, photorealistic, cinematic, natural" #@param {type:"string"}
input_image = "persona_ref.png" #@param {type:"string"}
width = 832 #@param {type:"integer"}
height = 480 #@param {type:"integer"}
seed = 7 #@param {type:"integer"}
steps = 20 #@param {type:"integer"}
guidance = 2.5 #@param {type:"number"}
import os, json, time, subprocess, urllib.request, urllib.parse, shutil
SERVER = f"http://127.0.0.1:{PORT}"; COMFY = "/content/ComfyUI"
DIFF=f"{COMFY}/models/diffusion_models"; TE=f"{COMFY}/models/text_encoders"; VAE=f"{COMFY}/models/vae"
for d in (DIFF,TE,VAE): os.makedirs(d, exist_ok=True)
FILES = {
    f"{DIFF}/flux1-kontext-dev-Q4_K_M.gguf":"https://huggingface.co/QuantStack/FLUX.1-Kontext-dev-GGUF/resolve/main/flux1-kontext-dev-Q4_K_M.gguf",
    f"{TE}/t5xxl_fp8_e4m3fn.safetensors":"https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors",
    f"{TE}/clip_l.safetensors":"https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors",
    f"{VAE}/ae.safetensors":"https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors",
}
for p,url in FILES.items():
    if os.path.exists(p): print(f"  exists: {os.path.basename(p)} ({os.path.getsize(p)//1048576} MB)")
    else:
        print(f"  downloading {os.path.basename(p)} ...")
        subprocess.run(["wget","-q","--show-progress","-O",p,url], check=True)
wf = {"prompt":{
  "12":{"class_type":"UnetLoaderGGUF","inputs":{"unet_name":"flux1-kontext-dev-Q4_K_M.gguf"}},
  "11":{"class_type":"DualCLIPLoader","inputs":{"clip_name1":"t5xxl_fp8_e4m3fn.safetensors","clip_name2":"clip_l.safetensors","type":"flux"}},
  "10":{"class_type":"VAELoader","inputs":{"vae_name":"ae.safetensors"}},
  "6":{"class_type":"CLIPTextEncode","inputs":{"text":scene_prompt,"clip":["11",0]}},
  "26":{"class_type":"FluxGuidance","inputs":{"conditioning":["6",0],"guidance":guidance}},
  "41":{"class_type":"LoadImage","inputs":{"image":input_image}},
  "40":{"class_type":"FluxKontextImageScale","inputs":{"image":["41",0]}},
  "39":{"class_type":"VAEEncode","inputs":{"pixels":["40",0],"vae":["10",0]}},
  "42":{"class_type":"ReferenceLatent","inputs":{"conditioning":["26",0],"latent":["39",0]}},
  "27":{"class_type":"EmptySD3LatentImage","inputs":{"width":width,"height":height,"batch_size":1}},
  "30":{"class_type":"ModelSamplingFlux","inputs":{"model":["12",0],"max_shift":1.15,"base_shift":0.5,"width":width,"height":height}},
  "25":{"class_type":"RandomNoise","inputs":{"noise_seed":seed}},
  "16":{"class_type":"KSamplerSelect","inputs":{"sampler_name":"euler"}},
  "17":{"class_type":"BasicScheduler","inputs":{"model":["30",0],"scheduler":"simple","steps":steps,"denoise":1.0}},
  "22":{"class_type":"BasicGuider","inputs":{"model":["30",0],"conditioning":["42",0]}},
  "13":{"class_type":"SamplerCustomAdvanced","inputs":{"noise":["25",0],"guider":["22",0],"sampler":["16",0],"sigmas":["17",0],"latent_image":["27",0]}},
  "8":{"class_type":"VAEDecode","inputs":{"samples":["13",0],"vae":["10",0]}},
  "9":{"class_type":"SaveImage","inputs":{"filename_prefix":"persona_scene","images":["8",0]}}
}}
pid=json.loads(urllib.request.urlopen(urllib.request.Request(f"{SERVER}/prompt",data=json.dumps(wf).encode(),headers={"Content-Type":"application/json"})).read())["prompt_id"]
print("submitted:",pid)
t0=time.time(); OUT=f"{COMFY}/output"; INP=f"{COMFY}/input"; LAST=None
while time.time()-t0<600:
    try: h=json.loads(urllib.request.urlopen(f"{SERVER}/history/{pid}").read())
    except: h={}
    if pid in h:
        if h[pid].get("status",{}).get("status_str")=="error": print("ERROR:",json.dumps(h[pid]["status"].get("messages",[]))[:500])
        for _,out in h[pid]["outputs"].items():
            for it in out.get("images",[]):
                q=urllib.parse.urlencode({"filename":it["filename"],"subfolder":it.get("subfolder",""),"type":"output"})
                data=urllib.request.urlopen(f"{SERVER}/view?{q}").read()
                dest=os.path.join(OUT,it["filename"]); open(dest,"wb").write(data); LAST=dest
                shutil.copy(dest, os.path.join(INP,"persona_scene.png"))
                print(f"DONE {time.time()-t0:.1f}s -> {dest} (also saved as input/persona_scene.png)")
        break
    time.sleep(2)
else: print("TIMEOUT")
